<a href="https://colab.research.google.com/github/hiten4/Day32AM/blob/main/Day_32_Part_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A — Decision Tree & Random Forest (Loan Approval System)


## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance


## 2. Create Synthetic Dataset (2000 records)

In [2]:
np.random.seed(42)

n = 2000

data = pd.DataFrame({
    'annual_income': np.random.randint(20000, 150000, n),
    'credit_score': np.random.randint(300, 850, n),
    'loan_amount': np.random.randint(5000, 50000, n),
    'employment_years': np.random.randint(0, 15, n),
    'debt_to_income': np.random.uniform(0.1, 0.6, n),
    'num_credit_cards': np.random.randint(1, 10, n)
})

data['approved'] = ((data['credit_score'] > 650) &
                    (data['debt_to_income'] < 0.4) &
                    (data['annual_income'] > 40000)).astype(int)

data.head()

,annual_income,credit_score,loan_amount,employment_years,debt_to_income,num_credit_cards,approved
0,141958,499,33884,5,0.361248,5,0
1,35795,836,29885,4,0.420512,4,0
2,20860,765,7708,6,0.493166,1,0
3,123694,388,23375,4,0.302554,3,0
4,148106,567,20088,9,0.101354,8,0


## 3. Train-Test Split

In [3]:
X = data.drop('approved', axis=1)
y = data['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4. Decision Tree (max_depth=4)

In [4]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

rules = export_text(dt, feature_names=list(X.columns))
print(rules)

|--- credit_score <= 650.50
|   |--- class: 0
|--- credit_score >  650.50
|   |--- debt_to_income <= 0.40
|   |   |--- annual_income <= 39680.50
|   |   |   |--- class: 0
|   |   |--- annual_income >  39680.50
|   |   |   |--- class: 1
|   |--- debt_to_income >  0.40
|   |   |--- class: 0



## 5. Random Forest (with RandomizedSearchCV)

In [5]:
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestClassifier(random_state=42)

search = RandomizedSearchCV(
    rf, param_dist, n_iter=5, cv=5, scoring='accuracy', random_state=42
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_


## 6. Model Evaluation

In [6]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

dt_metrics = evaluate(dt, X_test, y_test)
rf_metrics = evaluate(best_rf, X_test, y_test)

print("Decision Tree:", dt_metrics)
print("Random Forest:", rf_metrics)


Decision Tree: {'Accuracy': 0.9975, 'F1': 0.9941520467836257, 'ROC-AUC': np.float64(0.9941860465116279)}
Random Forest: {'Accuracy': 0.9975, 'F1': 0.9941520467836257, 'ROC-AUC': np.float64(0.9997778106947118)}


## 7. Feature Importance Comparison

In [7]:
rf_importance = pd.Series(best_rf.feature_importances_, index=X.columns)

perm = permutation_importance(best_rf, X_test, y_test, n_repeats=5, random_state=42)
perm_importance = pd.Series(perm.importances_mean, index=X.columns)

print("Default Importance:\n", rf_importance.sort_values(ascending=False))
print("\nPermutation Importance:\n", perm_importance.sort_values(ascending=False))


Default Importance:
 credit_score        0.578475
debt_to_income      0.261022
annual_income       0.133836
loan_amount         0.013164
num_credit_cards    0.007559
employment_years    0.005943
dtype: float64

Permutation Importance:
 credit_score        0.2520
debt_to_income      0.1555
annual_income       0.0430
employment_years    0.0015
loan_amount         0.0000
num_credit_cards    0.0000
dtype: float64


## 8. Recommendation

The Decision Tree model provides clear and interpretable rules, which is important for regulatory requirements. However, the Random Forest model achieves better overall performance in terms of accuracy, F1 score, and ROC-AUC due to reduced variance.

Therefore, a balanced approach is to use Random Forest for prediction while keeping Decision Tree for interpretability.